# Signatures comparison — new sorted-cell test cohort (CHESS-1333 / OD-128)

Companion to the published Figure 4 notebook (`Signatures comparison.ipynb`). The new sorted-cell cohort delivered with [Jira OD-128](https://bostongene.atlassian.net/browse/OD-128) serves as a true held-out **test** cohort, satisfying Reviewer 1's request to "Add new data of sorted cells to cell signature comparison" and the paper Methods commitment to ~75/25 train/test separation.

**Scope:** 16 of 20 FGES. The four rare-GOI FGES — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — are deferred to a separate rare-types notebook that uses 75/25 stratified holdouts on the original cohort (helpers ship in `signature_validation.benchmark.splits`).

**Random-FGES baseline:** v1 random gene lists are reused (loaded from `msigdb_gmt.pkl`) but rescored on the new cohort so ranks stay comparable.

**Where things land:** pickle and SVGs go to `/home/jovyan/SignValArticle/...` with a `_new_cohort` suffix; v1 outputs are not overwritten.

In [2]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
    load_new_cohort_expressions,
)
from signature_validation.benchmark.plotting import (
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import (
    compute_mapping_ssgseas,
    compute_out_table,
    fdr_correct_out,
)
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import cells_p

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

ModuleNotFoundError: No module named 'signature_validation.benchmark'

In [ ]:
REPO_ROOT = Path(".").resolve().parents[3]

NEW_ANNOT_PATH = Path(
    "/home/jovyan/projects/SignVal/Signature_validation/sorted_cells_to_check_all_annot.tsv"
)
EXPR_S3_PATH = "s3://bostongene-eurynome-exchange/raw_data/v2/expressions/osrp/"
V1_GMT_PICKLE = (
    REPO_ROOT
    / "Paper_Code_and_Figures"
    / "Figure_4"
    / "Cell_type_FGES_comparison"
    / "data"
    / "msigdb_gmt.pkl"
)

OUTPUT_DIR = Path("/home/jovyan/SignValArticle/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_SSGSEAS_PATH = OUTPUT_DIR / "mapping_ssgseas_new_cohort.pkl"
OUT_TSV_PATH = OUTPUT_DIR / "out_new_cohort.tsv"
HEATMAP_PATH = OUTPUT_DIR / "signature_heatmap_new_cohort.svg"

logger.info("new annotation:    {}", NEW_ANNOT_PATH)
logger.info("S3 expressions:    {}", EXPR_S3_PATH)
logger.info("v1 GMT pickle:     {}", V1_GMT_PICKLE)
logger.info("mapping_ssgseas:   {}", MAPPING_SSGSEAS_PATH)

In [ ]:
public_cells_annot = load_new_cohort_annotation(NEW_ANNOT_PATH)
public_cells_annot["Cell_type"].value_counts()

In [ ]:
public_cells_expr = load_new_cohort_expressions(public_cells_annot, path=EXPR_S3_PATH)
public_cells_expr.shape

In [ ]:
v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)

for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(v1_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"

msigdb_gmt = harmonize_gmt_to_index(v1_gmt, public_cells_expr.index)
logger.info("msigdb_gmt: {} FGES, {} signatures total", len(msigdb_gmt), sum(len(v) for v in msigdb_gmt.values()))

In [ ]:
mapping = build_mapping(annotation=public_cells_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, public_cells_annot)
logger.info(
    "in-scope: {} FGES; controls present in new cohort: {}",
    len(mapping),
    len(controls_present),
)
for sign, bucket in mapping.items():
    logger.info(
        "{}: GOI={}, Control={}, Deleted={}",
        sign,
        bucket["Goi"],
        len(bucket["Control"]),
        len(bucket["Deleted_controls"]),
    )

In [ ]:
mapping_ssgseas = compute_mapping_ssgseas(
    public_cells_expr=public_cells_expr,
    public_cells_annot=public_cells_annot,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
)

for sign in mapping_ssgseas:
    if sign in EXCLUDED_FGES_RARE:
        continue
    assert mapping_ssgseas[sign]["Goi"], f"{sign}: GOI cohort is empty"
    for ct, frame in mapping_ssgseas[sign]["Goi"].items():
        logger.info("{} GOI {}: {} samples", sign, ct, frame.shape[0])

with open(MAPPING_SSGSEAS_PATH, "wb") as fh:
    pickle.dump(mapping_ssgseas, fh, pickle.HIGHEST_PROTOCOL)
logger.info("wrote {}", MAPPING_SSGSEAS_PATH)

In [ ]:
out = compute_out_table(mapping_ssgseas, mapping, msigdb_gmt, controls_present)
out = fdr_correct_out(out, controls_present)
out.to_csv(OUT_TSV_PATH, sep="\t")
logger.info("wrote {} ({} rows × {} cols)", OUT_TSV_PATH, *out.shape)
out.head()

In [ ]:
plot_violin_per_source(mapping_ssgseas, save_dir=OUTPUT_DIR)
plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas,
    out_df=out,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
    annotation=public_cells_annot,
    controls_order=controls_present,
    palette={ct: cells_p[ct] for ct in controls_present if ct in cells_p},
    save_path=HEATMAP_PATH,
    short=True,
)
averaged = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
)
logger.info("plots saved under {}", OUTPUT_DIR)

## Rare cell types (out of scope here)

FGES whose GOI is rare in the new cohort — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — are deferred to a separate notebook authored by Nadezhda. That notebook reuses the original cohort (`/uftp2/.../cells_all_annotation.tsv` plus the Tonsillar Tfh / Mast / Endothelium_lymph patches), generates 10 stratified 75/25 holdouts via `signature_validation.benchmark.splits.stratified_holdout_indices` (stratified by BG-FGES score median × GOI/Control), scores ssGSEA on each test fold and aggregates with `aggregate_score_over_splits`. Rare cell types are starred on the resulting figures.